# PISA 2022 — 01 Extract everything

Source: <https://webfs.oecd.org/pisa2022/index.html> (PISA 2022 Data section, SPSS versions — the SAS files hold identical data).

This notebook:
1. downloads any missing zip,
2. unzips each SPSS `.SAV`,
3. converts every file to Parquet in chunks (the student files are too wide to load whole), stored as a folder of parts under 90 MB per dataset,
4. builds a **data dictionary** — every variable in every file with its label, type, value labels and % missing,
5. prints an inventory of what each file contains.

PISA's user-defined missing codes (95 Valid skip, 97 N/A, 98 Invalid, 99 No response, etc.) are converted to `NaN` on read.

**You do not need to run this notebook to use the data**: the Parquet folders and the data dictionary are already in the repo. Run it only to rebuild everything from the OECD source (~2 GB download, ~9 GB unzipped, ~10 min).

In [1]:
from pathlib import Path
import json, zipfile, subprocess, time
import pandas as pd, numpy as np
import pyarrow as pa, pyarrow.parquet as pq
import pyreadstat

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'requirements.txt').exists())  # repo root
RAW, PQ, OUT = ROOT/'data/raw', ROOT/'data/parquet', ROOT/'data'   # dictionary + inventory go in data/
for p in (RAW, PQ, OUT): p.mkdir(parents=True, exist_ok=True)
BASE = 'https://webfs.oecd.org/pisa2022/'

DATASETS = {
    'SCH_QQQ_SPSS': ('school',     'School questionnaire'),
    'STU_QQQ_SPSS': ('student',    'Student questionnaire (+ plausible values, weights, indices)'),
    'TCH_QQQ_SPSS': ('teacher',    'Teacher questionnaire'),
    'STU_COG_SPSS': ('cognitive',  'Cognitive item responses (raw + scored)'),
    'STU_TIM_SPSS': ('timing',     'Questionnaire timing'),
    'CRT_SPSS':     ('creative',   'Creative Thinking'),
    'FLT_SPSS':     ('finlit',     'Financial Literacy'),
    'escs_trend':   ('escs_trend', 'Rescaled ESCS for trend analysis (2012-2018)'),
}

In [2]:
# 1-2. download (resume-safe) + unzip
for stem in DATASETS:
    z = RAW/f'{stem}.zip'
    if not z.exists() or not zipfile.is_zipfile(z):
        subprocess.run(['curl', '-sSL', '-C', '-', '-o', str(z), BASE + z.name], check=True)
    with zipfile.ZipFile(z) as zf:
        for m in zf.infolist():
            if not (RAW/m.filename).exists():
                zf.extract(m, RAW)
        print(f'{stem:14s} {z.stat().st_size/1e6:8.1f} MB zip ->', [m.filename for m in zf.infolist()])

SCH_QQQ_SPSS        6.2 MB zip -> ['CY08MSP_SCH_QQQ.SAV']
STU_QQQ_SPSS      682.4 MB zip -> ['CY08MSP_STU_QQQ.SAV']
TCH_QQQ_SPSS       28.2 MB zip -> ['CY08MSP_TCH_QQQ.SAV']
STU_COG_SPSS      480.8 MB zip -> ['CY08MSP_STU_COG.SAV']
STU_TIM_SPSS      218.1 MB zip -> ['CY08MSP_STU_TIM.SAV']
CRT_SPSS          110.2 MB zip -> ['CY08MSP_CRT_COG.SAV']
FLT_SPSS          220.7 MB zip -> ['CY08MSP_FLT_COG.SAV', 'CY08MSP_FLT_QQQ.SAV', 'CY08MSP_FLT_TIM.SAV']
escs_trend         22.4 MB zip -> ['escs_trend.csv', 'ReadMe.txt']


In [3]:
# map each dataset to the extracted data files
files = {}
for stem, (key, _) in DATASETS.items():
    with zipfile.ZipFile(RAW/f'{stem}.zip') as zf:
        files[key] = [RAW/m.filename for m in zf.infolist()
                      if m.filename.lower().endswith(('.sav', '.csv'))]
files

{'school': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_SCH_QQQ.SAV')],
 'student': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_STU_QQQ.SAV')],
 'teacher': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_TCH_QQQ.SAV')],
 'cognitive': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_STU_COG.SAV')],
 'timing': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_STU_TIM.SAV')],
 'creative': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_CRT_COG.SAV')],
 'finlit': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_FLT_COG.SAV'),
  PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_FLT_QQQ.SAV'),
  PosixPath('/Users/alien/Desktop/Projects/personal/pisa-2022/data/raw/CY08MSP_FLT_TIM.SAV')],
 'escs_trend': [PosixPath('/Users/alien/Desktop/Projects/personal/pisa

## 3. Convert to Parquet + collect the dictionary

The Arrow schema comes from the SPSS metadata, not from the first chunk, so a column that happens to be all-missing in one chunk still gets a consistent type.

Each dataset is written as a **folder of Parquet parts** (`data/parquet/<file>/part-000.parquet`, ...), each under 90 MB so the data fits on GitHub. `pd.read_parquet(folder)` reads a folder exactly like a single file.

In [4]:
CHUNK = 20_000
PART_MB = 90   # GitHub rejects files over 100 MB, so each dataset is stored as a folder of parts

def split_parquet(single: Path, folder: Path):
    """Rewrite one Parquet file as folder/part-000.parquet, part-001.parquet, ... (each < PART_MB).
    Row order, schema and values are unchanged; the single file is removed only after the check passes."""
    pf = pq.ParquetFile(single)
    folder.mkdir(parents=True, exist_ok=True)
    part, size, writer = 0, 0, None
    for rg in range(pf.num_row_groups):
        g = pf.metadata.row_group(rg)
        rg_mb = sum(g.column(i).total_compressed_size for i in range(g.num_columns)) / 1e6
        if writer is None or size + rg_mb > PART_MB:
            if writer: writer.close(); part += 1
            writer = pq.ParquetWriter(folder/f'part-{part:03d}.parquet', pf.schema_arrow, compression='zstd')
            size = 0
        writer.write_table(pf.read_row_group(rg)); size += rg_mb
    writer.close()
    ds = pq.ParquetDataset(folder)
    assert sum(pq.ParquetFile(f).metadata.num_rows for f in ds.files) == pf.metadata.num_rows
    assert ds.schema.names == pf.schema_arrow.names
    single.unlink()

def parts(folder: Path): return sorted(folder.glob('part-*.parquet'))
def folder_mb(folder: Path): return sum(f.stat().st_size for f in parts(folder)) / 1e6

def convert_sav(src: Path, dst: Path):
    meta = pyreadstat.read_sav(str(src), metadataonly=True)[1]
    schema = pa.schema([(c, pa.string() if meta.readstat_variable_types[c] == 'string' else pa.float64())
                        for c in meta.column_names])
    nonnull = pd.Series(0, index=meta.column_names, dtype='int64')
    n = 0
    single = dst.with_suffix('.parquet')
    if not dst.exists() and not single.exists():
        writer = pq.ParquetWriter(single, schema, compression='zstd')
        for chunk, _ in pyreadstat.read_file_in_chunks(pyreadstat.read_sav, str(src), chunksize=CHUNK):
            nonnull += chunk.notna().sum()
            n += len(chunk)
            writer.write_table(pa.Table.from_pandas(chunk, schema=schema, preserve_index=False))
        writer.close()
        split_parquet(single, dst)
    else:  # already converted: recount missingness from parquet column stats
        if single.exists(): split_parquet(single, dst)
        for f in parts(dst):
            md = pq.ParquetFile(f).metadata
            n += md.num_rows
            for rg in range(md.num_row_groups):
                g = md.row_group(rg)
                for i in range(g.num_columns):
                    col = g.column(i)
                    nonnull[col.path_in_schema] += g.num_rows - col.statistics.null_count
    return meta, n, nonnull

def dict_rows(key, fname, meta, n, nonnull):
    vl = meta.variable_to_label
    for c in meta.column_names:
        labels = meta.value_labels.get(vl.get(c), {})
        yield dict(dataset=key, file=fname, variable=c,
                   label=meta.column_names_to_labels.get(c),
                   type=meta.readstat_variable_types[c],
                   pct_missing=round(100 * (1 - nonnull[c] / n), 2) if n else None,
                   n_value_labels=len(labels),
                   value_labels=json.dumps({str(k): v for k, v in labels.items()}) if labels else None)

inventory, dictionary = [], []
for key, srcs in files.items():
    for src in srcs:
        t = time.time()
        if src.suffix.lower() == '.sav':
            dst = PQ/src.stem
            meta, n, nonnull = convert_sav(src, dst)
            dictionary += list(dict_rows(key, src.name, meta, n, nonnull))
            ncol = len(meta.column_names)
        else:  # escs_trend ships as CSV
            df = pd.read_csv(src, low_memory=False)
            dst = PQ/src.stem
            if (PQ/f'{src.stem}.parquet').exists(): (PQ/f'{src.stem}.parquet').unlink()
            dst.mkdir(exist_ok=True)
            df.to_parquet(dst/'part-000.parquet', compression='zstd', index=False)
            n, ncol = df.shape
            dictionary += [dict(dataset=key, file=src.name, variable=c, label=None, type=str(df[c].dtype),
                                pct_missing=round(100*df[c].isna().mean(), 2), n_value_labels=0, value_labels=None)
                           for c in df.columns]
        cnt = pd.read_parquet(dst, columns=['CNT']).CNT if 'CNT' in pq.ParquetDataset(dst).schema.names else None
        inventory.append(dict(dataset=key, file=src.name, rows=n, columns=ncol,
                              countries=cnt.nunique() if cnt is not None else None,
                              sav_MB=round(src.stat().st_size/1e6, 1),
                              parquet_MB=round(folder_mb(dst), 1), parquet_parts=len(parts(dst)),
                              seconds=round(time.time()-t, 1)))
        print(inventory[-1])

inventory = pd.DataFrame(inventory)
dictionary = pd.DataFrame(dictionary)
inventory.to_csv(OUT/'inventory.csv', index=False)
dictionary.to_csv(OUT/'data_dictionary.csv', index=False)
inventory

{'dataset': 'school', 'file': 'CY08MSP_SCH_QQQ.SAV', 'rows': 21629, 'columns': 431, 'countries': 80, 'sav_MB': 19.7, 'parquet_MB': 5.0, 'parquet_parts': 1, 'seconds': 0.1}


{'dataset': 'student', 'file': 'CY08MSP_STU_QQQ.SAV', 'rows': 613744, 'columns': 1278, 'countries': 80, 'sav_MB': 2096.7, 'parquet_MB': 677.1, 'parquet_parts': 8, 'seconds': 0.3}
{'dataset': 'teacher', 'file': 'CY08MSP_TCH_QQQ.SAV', 'rows': 68054, 'columns': 543, 'countries': 18, 'sav_MB': 106.3, 'parquet_MB': 25.2, 'parquet_parts': 1, 'seconds': 0.0}


{'dataset': 'cognitive', 'file': 'CY08MSP_STU_COG.SAV', 'rows': 613744, 'columns': 5023, 'countries': 80, 'sav_MB': 3733.8, 'parquet_MB': 641.5, 'parquet_parts': 8, 'seconds': 1.0}
{'dataset': 'timing', 'file': 'CY08MSP_STU_TIM.SAV', 'rows': 613744, 'columns': 172, 'countries': 80, 'sav_MB': 663.7, 'parquet_MB': 251.2, 'parquet_parts': 3, 'seconds': 0.0}
{'dataset': 'creative', 'file': 'CY08MSP_CRT_COG.SAV', 'rows': 499843, 'columns': 301, 'countries': 63, 'sav_MB': 398.4, 'parquet_MB': 125.0, 'parquet_parts': 2, 'seconds': 0.1}


{'dataset': 'finlit', 'file': 'CY08MSP_FLT_COG.SAV', 'rows': 97983, 'columns': 3637, 'countries': 20, 'sav_MB': 505.7, 'parquet_MB': 117.7, 'parquet_parts': 2, 'seconds': 0.3}
{'dataset': 'finlit', 'file': 'CY08MSP_FLT_QQQ.SAV', 'rows': 97983, 'columns': 1192, 'countries': 20, 'sav_MB': 272.4, 'parquet_MB': 77.8, 'parquet_parts': 1, 'seconds': 0.1}
{'dataset': 'finlit', 'file': 'CY08MSP_FLT_TIM.SAV', 'rows': 97983, 'columns': 172, 'countries': 20, 'sav_MB': 117.8, 'parquet_MB': 46.2, 'parquet_parts': 1, 'seconds': 0.0}


{'dataset': 'escs_trend', 'file': 'escs_trend.csv', 'rows': 1447787, 'columns': 10, 'countries': None, 'sav_MB': 77.3, 'parquet_MB': 19.6, 'parquet_parts': 1, 'seconds': 0.7}


,dataset,file,rows,columns,countries,sav_MB,parquet_MB,parquet_parts,seconds
0,school,CY08MSP_SCH_QQQ.SAV,21629,431,80.0,19.7,5.0,1,0.1
1,student,CY08MSP_STU_QQQ.SAV,613744,1278,80.0,2096.7,677.1,8,0.3
2,teacher,CY08MSP_TCH_QQQ.SAV,68054,543,18.0,106.3,25.2,1,0.0
3,cognitive,CY08MSP_STU_COG.SAV,613744,5023,80.0,3733.8,641.5,8,1.0
4,timing,CY08MSP_STU_TIM.SAV,613744,172,80.0,663.7,251.2,3,0.0
5,creative,CY08MSP_CRT_COG.SAV,499843,301,63.0,398.4,125.0,2,0.1
6,finlit,CY08MSP_FLT_COG.SAV,97983,3637,20.0,505.7,117.7,2,0.3
7,finlit,CY08MSP_FLT_QQQ.SAV,97983,1192,20.0,272.4,77.8,1,0.1
8,finlit,CY08MSP_FLT_TIM.SAV,97983,172,20.0,117.8,46.2,1,0.0
9,escs_trend,escs_trend.csv,1447787,10,NaN,77.3,19.6,1,0.7


## 4. What's in each file

In [5]:
pd.set_option('display.max_colwidth', 90, 'display.width', 200)
print(f"{len(dictionary):,} variables across {inventory.file.nunique()} files, "
      f"{inventory.rows.sum():,} rows total\n")
dictionary.groupby('dataset').agg(variables=('variable','size'),
                                  fully_missing=('pct_missing', lambda s: (s==100).sum()),
                                  median_pct_missing=('pct_missing','median'))

12,759 variables across 10 files, 4,172,494 rows total



,variables,fully_missing,median_pct_missing
dataset,,,
cognitive,5023,6,90.780
creative,301,5,88.740
escs_trend,10,0,0.000
finlit,5001,3,87.540
school,431,0,12.460
student,1278,0,51.010
teacher,543,0,5.090
timing,172,0,12.105


In [6]:
# Variable families by name prefix: ST=student questionnaire item, SC=school item, TC=teacher item,
# PV*=plausible values (scores), W_*=weights, IC/FL/PA/EC=optional questionnaires, uppercase words=derived indices
def family(v):
    import re
    if v.startswith('PV') : return 'plausible values (scores)'
    if v.startswith('W_') : return 'weights / BRR replicates'
    if re.match(r'^(ST|SC|TC|IC|FL|EC|PA|WB|CR)\d', v): return f'questionnaire item ({v[:2]})'
    if re.match(r'^[A-Z]{2,4}\d{2,}', v) or re.match(r'^[CDMPRSF]\d{3}', v): return 'cognitive item / timing'
    return 'id / derived index / admin'

dictionary['family'] = dictionary.variable.map(family)
dictionary.pivot_table(index='family', columns='dataset', values='variable', aggfunc='size', fill_value=0)

dataset,cognitive,creative,escs_trend,finlit,school,student,teacher,timing
family,,,,,,,,
cognitive item / timing,3717,55,0,2338,0,1,1,0
id / derived index / admin,40,206,10,209,69,148,79,27
plausible values (scores),0,40,0,30,0,110,0,0
questionnaire item (CR),1266,0,0,1266,0,0,0,0
questionnaire item (FL),0,0,0,107,0,93,0,14
questionnaire item (IC),0,0,0,116,0,101,0,15
questionnaire item (PA),0,0,0,145,0,145,0,0
questionnaire item (SC),0,0,0,0,359,0,0,0
questionnaire item (ST),0,0,0,601,0,516,0,91


In [7]:
# Derived indices in the student file (the scaled constructs most analyses use)
stu = dictionary[(dictionary.dataset=='student') & (dictionary.family=='id / derived index / admin')]
stu[['variable','label','pct_missing']].to_string(index=False)[:12000]

"    variable                                                                                               label  pct_missing\n         CNT                                                                            Country code 3-character         0.00\n     CNTRYID                                                                                  Country Identifier         0.00\n    CNTSCHID                                                                                     Intl. School ID         0.00\n    CNTSTUID                                                                                    Intl. Student ID         0.00\n         CYC                              PISA Assessment Cycle (2 digits + 2 character Assessment type - MS/FT)         0.00\n      NatCen                                                                        National Centre 6-digit Code         0.00\n     STRATUM                                                  Stratum ID 5-character (cnt + original stratum I

In [8]:
# Plausible values available per domain (student, creative thinking, financial literacy files)
pv = dictionary[dictionary.family=='plausible values (scores)'].copy()
pv['domain'] = pv.variable.str.replace(r'^PV\d+', '', regex=True)
pv.groupby(['dataset','domain']).agg(n_pv=('variable','size'), label=('label','first')).reset_index()

,dataset,domain,n_pv,label
0,creative,CRTH_NC,10,Plausible Value 1 in Creative Thinking (Number Correct)
1,creative,MATC,10,Plausible Value 1 in Mathematics (CRT version)
2,creative,REAC,10,Plausible Value 1 in Reading (CRT version)
3,creative,SCIC,10,Plausible Value 1 in Science (CRT version)
4,finlit,FLIT,10,Plausible Value 1 in Financial Literacy
5,finlit,MATH,10,Plausible Value 1 in Mathematics
6,finlit,READ,10,Plausible Value 1 in Reading
7,student,MATH,10,Plausible Value 1 in Mathematics
8,student,MCCR,10,Plausible Value 1 in Content Subscale of Mathematics - Change and Relationships
9,student,MCQN,10,Plausible Value 1 in Content Subscale of Mathematics - Quantity


Next: `02_explore.ipynb` for the weighted analysis.